## Model Training for SSD

In [1]:
!pip install ultralytics
!pip install tqdm
!pip install torchmetrics
!pip install pycocotools
!pip install faster-coco-eval

In [2]:
import os
import time
from pathlib import Path
from collections import Counter
import torch
from torch.utils.data import Dataset, DataLoader
import cv2
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
from torchvision.models.detection import ssd300_vgg16
from tqdm import tqdm
from torchmetrics.detection.mean_ap import MeanAveragePrecision

### If running SSD training locally



In [ ]:
root = Path("../../final_unified_dataset")
print("Dataset exists:", root.exists())
print("Train images:", (root / "images/train").exists())
print("Val images:", (root / "images/val").exists())
print("Test images:", (root / "images/test").exists())
print("Train labels:", (root / "labels/train").exists())
print("Val labels:", (root / "labels/val").exists())
print("Test labels:", (root / "labels/test").exists())
print("YAML exists:", (root / "dataset.yaml").exists())
print(Path("../../final_unified_dataset/dataset.yaml").read_text())

for split in ["train", "val", "test"]:
    img_dir = root / "images" / split
    lbl_dir = root / "labels" / split

    num_images = len(list(img_dir.glob("*.*"))) if img_dir.exists() else 0
    num_labels = len(list(lbl_dir.glob("*.txt"))) if lbl_dir.exists() else 0

    print(f"\n{split.capitalize()} split:")
    print(f"  Images: {num_images}")
    print(f"  Labels: {num_labels}")
    


Dataset exists: True
Train images: True
Val images: True
Test images: True
Train labels: True
Val labels: True
Test labels: True
YAML exists: True
path: C:\Users\skipp\Documents\Obsidian Vault\SUTD\Term 6 (Skipper)\Computational Data Science\Proj\test\CDS_2026Spring_project\final_unified_dataset
train: images/train
val:   images/val
test:  images/test

nc: 2
names: ['person', 'bag']


Train split:
  Images: 4039
  Labels: 4039

Val split:
  Images: 504
  Labels: 504

Test split:
  Images: 507
  Labels: 507


In [5]:
base = "../../final_unified_dataset"

for split in ["train", "val", "test"]:
    label_dir = os.path.join(base, "labels", split)
    counter = Counter()
    for file in os.listdir(label_dir):
        if file.endswith(".txt"):
            with open(os.path.join(label_dir, file)) as f:
                for line in f:
                    parts = line.strip().split()
                    if parts:
                        counter[int(parts[0])] += 1
    print(f"{split}: person={counter[0]}, bag={counter[1]}, ratio={counter[1]/counter[0]:.1f}x")
    
    
    

train: person=18649, bag=5488, ratio=0.3x
val: person=2211, bag=660, ratio=0.3x
test: person=2336, bag=698, ratio=0.3x


In [6]:
# SSD model setup (local)
class SSDDataset(Dataset):
    def __init__(self, root, split="train"):
        self.root = Path(root)
        self.img_dir = self.root / f"images/{split}"
        self.label_dir = self.root / f"labels/{split}"

        self.images = list(self.img_dir.glob("*.jpg"))

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        label_path = self.label_dir / (img_path.stem + ".txt")

        image = cv2.imread(str(img_path))
        h, w = image.shape[:2]
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        boxes = []
        labels = []

        if label_path.exists():
            with open(label_path) as f:
                for line in f:
                    cls, x, y, bw, bh = map(float, line.strip().split())

                    x_min = (x - bw/2) * w
                    y_min = (y - bh/2) * h
                    x_max = (x + bw/2) * w
                    y_max = (y + bh/2) * h

                    boxes.append([x_min, y_min, x_max, y_max])
                    labels.append(int(cls) + 1)
                    # +1 because 0 = background in SSD

        boxes = torch.tensor(boxes, dtype=torch.float32)
        labels = torch.tensor(labels, dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels
        }

        image = torch.tensor(image / 255.0, dtype=torch.float32).permute(2, 0, 1)

        return image, target

In [ ]:
# SSD Training (Local)
root = "../../final_unified_dataset"

train_dataset = SSDDataset(root, "train")
val_dataset = SSDDataset(root, "val")

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True,
                          collate_fn=lambda x: tuple(zip(*x)))
val_loader = DataLoader(val_dataset, batch_size=2, shuffle=False,
                        collate_fn=lambda x: tuple(zip(*x)))

device = torch.device("cpu")
print("Using device:", device)

# Load model
model = ssd300_vgg16(weights="DEFAULT")
model.head.classification_head.num_classes = 3
model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
def compute_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    inter_area = max(0, x2 - x1) * max(0, y2 - y1)
    box1_area = (box1[2]-box1[0]) * (box1[3]-box1[1])
    box2_area = (box2[2]-box2[0]) * (box2[3]-box2[1])
    union_area = box1_area + box2_area - inter_area
    return inter_area / union_area if union_area > 0 else 0

def batch_precision_recall(outputs, targets, iou_threshold=0.5):
    correct = 0
    total_pred = 0
    total_gt = 0

    for out, tgt in zip(outputs, targets):
        pred_boxes = out['boxes'].detach().cpu()
        gt_boxes = tgt['boxes'].detach().cpu()

        total_pred += len(pred_boxes)
        total_gt += len(gt_boxes)

        for p in pred_boxes:
            if any(compute_iou(p, g) > iou_threshold for g in gt_boxes):
                correct += 1

    precision = correct / total_pred if total_pred > 0 else 0
    recall = correct / total_gt if total_gt > 0 else 0
    return precision, recall

start_time_total = time.time()
epochs = 20

for epoch in range(epochs):
    epoch_start = time.time()
    model.train()
    total_loss = 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")

    for images, targets in pbar:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        loss = sum(loss_dict.values())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        pbar.set_postfix({"batch_loss": f"{loss.item():.3f}"})

    model.eval()
    precisions = []
    recalls = []

    with torch.no_grad():
        for images, targets in val_loader:
            images = [img.to(device) for img in images]
            outputs = model(images)

            p, r = batch_precision_recall(outputs, targets)
            precisions.append(p)
            recalls.append(r)

    epoch_precision = sum(precisions) / len(precisions)
    epoch_recall = sum(recalls) / len(recalls)
    epoch_time = time.time() - epoch_start

    print(f"Epoch {epoch+1}/{epochs} | Total Loss: {total_loss:.4f} | "
          f"Precision: {epoch_precision:.3f} | Recall: {epoch_recall:.3f} | "
          f"Epoch Time: {epoch_time:.2f}s")

total_time = time.time() - start_time_total
print(f"\nTotal training time: {total_time/60:.2f} minutes")

In [ ]:
# Saving model 
save_path = "ssd_trained_model_2.pth"

torch.save(model.state_dict(), save_path)

print(f"Model saved to {save_path}")

In [ ]:
# Load + evaluate model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = ssd300_vgg16(weights="DEFAULT")
model.head.classification_head.num_classes = 3  # 2 classes + background

save_path = "../../runs/detect/SSDv3_COCO_final/train2//weights/best.pth"
model.load_state_dict(torch.load(save_path, map_location=device))

model.to(device)
model.eval()

print(f"Model loaded from {save_path}")

Model loaded from ../../runs/detect/SSDv3_COCO_final/train2//weights/best.pth


In [ ]:
# SSD Inference and Visualization

root = Path("../../final_unified_dataset")


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = ssd300_vgg16(weights="DEFAULT")
model.head.classification_head.num_classes = 3

save_path = "../../runs/detect/SSDv3_COCO_final/train2//weights/best.pth"
model.load_state_dict(torch.load(save_path, map_location=device))

model.to(device)
model.eval()
print("Loaded SSD model for inference.")

test_image_path = root / "images/train/0a23e1fda36faaab.jpg"

img_cv2 = cv2.imread(str(test_image_path))

if img_cv2 is None:
    raise FileNotFoundError(f"Image not found at {test_image_path}")

img_rgb = cv2.cvtColor(img_cv2, cv2.COLOR_BGR2RGB)

img_tensor = torch.tensor(img_rgb / 255.0, dtype=torch.float32)\
                .permute(2, 0, 1)\
                .unsqueeze(0)\
                .to(device)

with torch.no_grad():
    outputs = model(img_tensor)


def draw_boxes(image_path, outputs, conf_threshold=0.25):
    img = Image.open(image_path).copy()
    draw = ImageDraw.Draw(img)

    class_names = {1: "person", 2: "bag"}

    boxes = outputs[0]['boxes'].cpu()
    labels = outputs[0]['labels'].cpu()
    scores = outputs[0]['scores'].cpu()

    for box, label, score in zip(boxes, labels, scores):
        label = int(label)

        if score < conf_threshold:
            continue

        x1, y1, x2, y2 = map(int, box)

        color = "cyan" if label == 1 else "orange"
        name = class_names.get(label, "unknown")

        draw.rectangle([x1, y1, x2, y2], outline=color, width=3)
        draw.text((x1, y1-15), f"{name} {score:.2f}", fill=color)

    return img
img_result = draw_boxes(test_image_path, outputs, conf_threshold=0.25)

plt.figure(figsize=(10, 8))
plt.imshow(img_result)
plt.axis("off")
plt.show()

label_path = root / "labels/train/0a23e1fda36faaab.txt"

if label_path.exists():
    print(f"\nGround-truth annotations from {label_path}:")
    with open(label_path) as f:
        for i, line in enumerate(f.readlines()):
            cls, cx, cy, bw, bh = map(float, line.strip().split())
            cls_name = "person" if int(cls) == 0 else "bag"
            print(f"{i+1}. Class: {cls_name}, Center: ({cx:.3f},{cy:.3f}), Size: {bw:.3f}x{bh:.3f}")
else:
    print("No ground-truth labels found for this image.")

Loaded SSD model for inference.


FileNotFoundError: Image not found at ..\..\final_unified_dataset\images\train\0a23e1fda36faaab.jpg

In [10]:
from pathlib import Path
import torch
import time
from torchvision.models.detection import ssd300_vgg16
from torch.utils.data import DataLoader
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from torchvision.ops import box_iou
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def compute_precision(outputs, targets, iou_thresh=0.5, conf_thresh=0.5):
    tp, fp = 0, 0

    for pred, gt in zip(outputs, targets):
        scores = pred["scores"]
        boxes = pred["boxes"]

        keep = scores > conf_thresh
        pred_boxes = boxes[keep]

        gt_boxes = gt["boxes"]

        if len(pred_boxes) == 0:
            continue
        if len(gt_boxes) == 0:
            fp += len(pred_boxes)
            continue

        ious = box_iou(pred_boxes, gt_boxes)

        matched_gt = set()
        for i in range(len(pred_boxes)):
            max_iou, idx = ious[i].max(0)
            if max_iou > iou_thresh and idx.item() not in matched_gt:
                tp += 1
                matched_gt.add(idx.item())
            else:
                fp += 1

    precision = tp / (tp + fp + 1e-6)
    return precision

root = Path("../../final_unified_dataset")

val_dataset = SSDDataset(root, "val")
val_loader = DataLoader(
    val_dataset,
    batch_size=2,
    shuffle=False,
    collate_fn=lambda x: tuple(zip(*x))
)

model = ssd300_vgg16(weights="DEFAULT")
model.head.classification_head.num_classes = 3

model.load_state_dict(torch.load(
    "../../runs/detect/SSDv3_COCO_final/train2/weights/best.pth",
    map_location=device
))

model.to(device)
model.eval()
map_metric = MeanAveragePrecision(iou_thresholds=[0.5], class_metrics=True)
inference_times = []

all_outputs = []
all_targets = []

with torch.no_grad():
    for images, targets in tqdm(val_loader, desc="Evaluating SSD"):
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        if device.type == "cuda":
            start = torch.cuda.Event(enable_timing=True)
            end = torch.cuda.Event(enable_timing=True)

            start.record()
            outputs = model(images)
            end.record()

            torch.cuda.synchronize()
            inference_times.append(start.elapsed_time(end))
        else:
            t0 = time.time()
            outputs = model(images)
            t1 = time.time()
            inference_times.append((t1 - t0) * 1000)

        all_outputs.extend([{k: v.cpu() for k, v in o.items()} for o in outputs])
        all_targets.extend([{k: v.cpu() for k, v in t.items()} for t in targets])

        map_metric.update(outputs, targets)


map_results = map_metric.compute()
precision = compute_precision(all_outputs, all_targets, iou_thresh=0.5, conf_thresh=0.5)

ssd_results = {
    "mAP50": map_results["map_50"].item(),
    "mAP50-95": map_results["map"].item(),
    "mean_recall": map_results["mar_100"].item(),
    "precision@0.5": precision,
    "inference_ms": sum(inference_times) / len(inference_times),
}

print("\nSSD evaluation results:")
for k, v in ssd_results.items():
    print(f"  {k}: {v:.4f}")

Evaluating SSD: 100%|██████████| 252/252 [01:15<00:00,  3.34it/s]



SSD evaluation results:
  mAP50: 0.3294
  mAP50-95: 0.3294
  mean_recall: 0.6158
  precision@0.5: 0.7694
  inference_ms: 287.7910
